## Hybrid grid search
Similarly to the **Authentic-only grid search** this script trains the classifier across all eight configurations in the hyperparameter grid, with each configuration evaluated under 5-fold cross-validation on the tuning split (seed 42). Configurations are trained one at a time by setting `CONFIG_IDX` and running the script; completed folds are tracked in a per-configuration results log and skipped on rerun, which makes the procedure resumable across Colab sessions.

Final results across configurations is performed by a separate script.

In [1]:
from google.colab import drive
drive.mount('/content/drive/')

import os
import json
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# Reproducibility
SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

print("TF version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices("GPU"))

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).
TF version: 2.20.0
GPU available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [2]:
HYPERPARAM_GRID = [
    {"lr": 3e-5, "unfrozen_layers": 40, "dropout": (0.3, 0.6)},
    {"lr": 3e-5, "unfrozen_layers": 40, "dropout": (0.4, 0.5)},
    {"lr": 3e-5, "unfrozen_layers": 60, "dropout": (0.3, 0.6)},
    {"lr": 3e-5, "unfrozen_layers": 60, "dropout": (0.4, 0.5)},
    {"lr": 5e-5, "unfrozen_layers": 40, "dropout": (0.3, 0.6)},
    {"lr": 5e-5, "unfrozen_layers": 40, "dropout": (0.4, 0.5)},
    {"lr": 5e-5, "unfrozen_layers": 60, "dropout": (0.3, 0.6)},
    {"lr": 5e-5, "unfrozen_layers": 60, "dropout": (0.4, 0.5)},
]

# Tuning split (seed 42). The evaluation split lives under authentic_split_cv_eval and is used in a separate script.
CV_DIR = os.path.join("/content/drive/MyDrive", "authentic_split_cv_tuning")
synAug_PATH = os.path.join("/content/drive/MyDrive", "augmentation_data", "tank_classifier_v3.keras")
RESULTS_ROOT = os.path.join("/content/drive/MyDrive", "authentic_training_cv_grid", "model3")

os.makedirs(RESULTS_ROOT, exist_ok=True)

# Sanity check: print the image counts in each fold/split before any training starts, so missing or malformed folds are caught early.
print("Verifying dataset:")
for fold in range(5):
    fold_dir = os.path.join(CV_DIR, f"fold_{fold}")
    for split in ("train", "val", "test"):
        path = os.path.join(fold_dir, split)
        if os.path.isdir(path):
            total = sum(
                len(os.listdir(os.path.join(path, cls)))
                for cls in os.listdir(path)
                if os.path.isdir(os.path.join(path, cls))
            )
            print(f"  fold_{fold}/{split}: {total} images")
        else:
            print(f"  fold_{fold}/{split}: MISSING")

print("\nHyperparameter grid")
for i, cfg in enumerate(HYPERPARAM_GRID):
    print(
        f"  Config {i}: lr={cfg['lr']}, layers={cfg['unfrozen_layers']}, "
        f"dropout={cfg['dropout']}"
    )

Verifying dataset:
  fold_0/train: 516 images
  fold_0/val: 173 images
  fold_0/test: 173 images
  fold_1/train: 516 images
  fold_1/val: 173 images
  fold_1/test: 173 images
  fold_2/train: 517 images
  fold_2/val: 173 images
  fold_2/test: 172 images
  fold_3/train: 517 images
  fold_3/val: 173 images
  fold_3/test: 172 images
  fold_4/train: 517 images
  fold_4/val: 173 images
  fold_4/test: 172 images

Hyperparameter grid
  Config 0: lr=3e-05, layers=40, dropout=(0.3, 0.6)
  Config 1: lr=3e-05, layers=40, dropout=(0.4, 0.5)
  Config 2: lr=3e-05, layers=60, dropout=(0.3, 0.6)
  Config 3: lr=3e-05, layers=60, dropout=(0.4, 0.5)
  Config 4: lr=5e-05, layers=40, dropout=(0.3, 0.6)
  Config 5: lr=5e-05, layers=40, dropout=(0.4, 0.5)
  Config 6: lr=5e-05, layers=60, dropout=(0.3, 0.6)
  Config 7: lr=5e-05, layers=60, dropout=(0.4, 0.5)


In [3]:
# Select configuration index (0..7)
CONFIG_IDX = 1

config = HYPERPARAM_GRID[CONFIG_IDX]
LR = config["lr"]
UNFROZEN_LAYERS = config["unfrozen_layers"]
DROPOUT_1, DROPOUT_2 = config["dropout"]

CONFIG_NAME = (
    f"config_{CONFIG_IDX}_lr{LR:.0e}_lay{UNFROZEN_LAYERS}"
    f"_drop{DROPOUT_1}_{DROPOUT_2}"
).replace(".", "p")

print(f"Configuration {CONFIG_IDX}: {CONFIG_NAME}")
print(f"  LR:              {LR}")
print(f"  Unfrozen layers: {UNFROZEN_LAYERS}")
print(f"  Dropout:         ({DROPOUT_1}, {DROPOUT_2})")

CONFIG_RESULTS_DIR = os.path.join(RESULTS_ROOT, CONFIG_NAME)
RESULTS_LOG = os.path.join(CONFIG_RESULTS_DIR, "results.json")
os.makedirs(CONFIG_RESULTS_DIR, exist_ok=True)

if os.path.exists(RESULTS_LOG):
    with open(RESULTS_LOG, "r") as f:
        config_results = json.load(f)
    completed_folds = sorted(config_results.keys())
    print(f"\nAlready completed folds: {completed_folds}")
else:
    config_results = {}
    completed_folds = []

img_size = (682, 1024)
batch_size = 4
num_classes = 3
AUTOTUNE = tf.data.AUTOTUNE

# Load synAug model weights once per configuration
print(f"\nLoading synAug from: {synAug_PATH}")
model2 = keras.models.load_model(synAug_PATH)
print(f"synAug model loaded. Layers: {len(model2.layers)}")

# Locate synAug's EfficientNetB4 submodel and dense layers
model2_base = None
for layer in model2.layers:
    if isinstance(layer, keras.Model) and "efficientnet" in layer.name.lower():
        model2_base = layer
        break
if model2_base is None:
    raise ValueError("Could not find EfficientNetB4 base in synAug")
print(f"Found synAug base: {model2_base.name}")

model2_dense = None
model2_output_dense = None
for layer in model2.layers:
    if isinstance(layer, layers.Dense):
        if layer.units == 128:
            model2_dense = layer
        elif layer.units == num_classes:
            model2_output_dense = layer

print(f"Dense(128): {model2_dense.name if model2_dense else 'NOT FOUND'}")
print(f"Output Dense({num_classes}): {model2_output_dense.name if model2_output_dense else 'NOT FOUND'}")

# Save synAug weights then free the model object
model2_base_weights = model2_base.get_weights()
model2_dense_weights = model2_dense.get_weights() if model2_dense else None
model2_output_weights = model2_output_dense.get_weights() if model2_output_dense else None

del model2
keras.backend.clear_session()
print("synAug weights saved; original model dereferenced.")

for FOLD_IDX in range(5):
    fold_key = f"fold_{FOLD_IDX}"
    if fold_key in completed_folds:
        print(f"\nSkipping {fold_key} (already completed)")
        continue

    print(f"\nTraining {CONFIG_NAME} on {fold_key}")

    dataset_dir = os.path.join(CV_DIR, fold_key)
    train_dir = os.path.join(dataset_dir, "train")
    val_dir = os.path.join(dataset_dir, "val")
    test_dir = os.path.join(dataset_dir, "test")

    val_ds = keras.utils.image_dataset_from_directory(
        val_dir,
        labels="inferred",
        label_mode="int",
        image_size=img_size,
        batch_size=batch_size,
        shuffle=False,
        crop_to_aspect_ratio=True,
    )
    test_ds = keras.utils.image_dataset_from_directory(
        test_dir,
        labels="inferred",
        label_mode="int",
        image_size=img_size,
        batch_size=batch_size,
        shuffle=False,
        crop_to_aspect_ratio=True,
    )

    class_names = val_ds.class_names
    val_ds = val_ds.prefetch(AUTOTUNE)
    test_ds = test_ds.prefetch(AUTOTUNE)

    data_augmentation = keras.Sequential([
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.05),
        layers.RandomZoom(0.10),
        layers.RandomTranslation(0.05, 0.05),
        layers.RandomBrightness(0.2),
        layers.RandomContrast(0.2),
    ])

    t90_aug = keras.Sequential([
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.10),
        layers.RandomZoom(0.15),
        layers.RandomTranslation(0.10, 0.10),
        layers.RandomBrightness(0.3),
        layers.RandomContrast(0.3),
    ])

    def load_class_ds(class_path, label, img_size, augmentation):
        ds = keras.utils.image_dataset_from_directory(
            class_path,
            labels=None,
            image_size=img_size,
            batch_size=None,
            shuffle=True,
            seed=SEED,
            crop_to_aspect_ratio=True,
        )
        return ds.map(lambda img: (augmentation(img, training=True), tf.cast(label, tf.int32)))

    t72_ds = load_class_ds(os.path.join(train_dir, "t72nolabel"), 0, img_size, data_augmentation)
    t80_ds = load_class_ds(os.path.join(train_dir, "t80nolabel"), 1, img_size, data_augmentation)
    t90_ds = load_class_ds(os.path.join(train_dir, "t90nolabel"), 2, img_size, t90_aug)

    train_ds = (
        t72_ds
        .concatenate(t80_ds)
        .concatenate(t90_ds)
        .shuffle(buffer_size=1000, seed=SEED)
        .batch(batch_size)
        .prefetch(AUTOTUNE)
    )

    base_model = keras.applications.EfficientNetB4(
        input_shape=img_size + (3,),
        include_top=False,
        weights=None,
    )

    # Initialize base EfficientNetB4 with weights from synAug to preserve the learned visual features.
    base_model.set_weights(model2_base_weights)

    base_model.trainable = True
    for layer in base_model.layers[:-UNFROZEN_LAYERS]:
        layer.trainable = False

    trainable_count = sum(layer.trainable for layer in base_model.layers)
    print(f"Trainable layers in base_model: {trainable_count}/{len(base_model.layers)}")

    inputs = keras.Input(shape=img_size + (3,))
    x = keras.applications.efficientnet.preprocess_input(inputs)
    x = base_model(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(DROPOUT_1)(x)
    dense_layer = layers.Dense(128, activation="relu", kernel_regularizer=keras.regularizers.l2(1e-2))
    x = dense_layer(x)
    x = layers.Dropout(DROPOUT_2)(x)
    output_layer = layers.Dense(num_classes, activation="softmax")
    outputs = output_layer(x)

    model = keras.Model(inputs, outputs, name=f"m3_{CONFIG_NAME}_fold{FOLD_IDX}")

    # transfer the trained classifier weights from the synAug model to the hybrid model's classifier.
    if model2_dense_weights is not None:
        dense_layer.set_weights(model2_dense_weights)
        print("Transferred Dense(128) weights from synAug.")
    if model2_output_weights is not None:
        output_layer.set_weights(model2_output_weights)
        print(f"Transferred Dense({num_classes}) weights from synAug.")

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=LR),
        loss=keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
        metrics=["accuracy"],
    )

    def to_one_hot(image, label):
        return image, tf.one_hot(label, num_classes)

    train_ds_oh = train_ds.map(to_one_hot, num_parallel_calls=AUTOTUNE)
    val_ds_oh = val_ds.map(to_one_hot, num_parallel_calls=AUTOTUNE)
    test_ds_oh = test_ds.map(to_one_hot, num_parallel_calls=AUTOTUNE)

    class_counts = {
        i: len(os.listdir(os.path.join(train_dir, cls)))
        for i, cls in enumerate(class_names)
    }
    total = sum(class_counts.values())
    class_weight = {i: total / (len(class_counts) * count) for i, count in class_counts.items()}
    print(f"Class weights: {class_weight}")

    CHECKPOINT_DIR = os.path.join(CONFIG_RESULTS_DIR, f"checkpoints_fold{FOLD_IDX}")
    HISTORY_PATH = os.path.join(CONFIG_RESULTS_DIR, f"history_fold{FOLD_IDX}.json")
    MODEL_PATH = os.path.join(CONFIG_RESULTS_DIR, f"model_fold{FOLD_IDX}.keras")
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)

    class SaveHistoryCallback(keras.callbacks.Callback):
        def __init__(self, filepath):
            self.filepath = filepath
        def on_epoch_end(self, epoch, logs=None):
            with open(self.filepath, "w") as f:
                json.dump(self.model.history.history, f)

    callbacks = [
        keras.callbacks.EarlyStopping(
            monitor="val_accuracy", 
            patience=15, 
            restore_best_weights=True
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", 
            factor=0.5, 
            patience=3, 
            min_lr=1e-7
        ),
        keras.callbacks.ModelCheckpoint(
            filepath=os.path.join(CHECKPOINT_DIR, "epoch_{epoch:02d}_val{val_accuracy:.3f}.keras"), 
            monitor="val_accuracy", 
            save_best_only=True, 
            verbose=0
        ),
        SaveHistoryCallback(filepath=HISTORY_PATH),
    ]

    history = model.fit(
        train_ds_oh,
        validation_data=val_ds_oh,
        epochs=50,
        class_weight=class_weight,
        callbacks=callbacks,
        verbose=2,
    )

    # Save, clear, and reload the persisted model before evaluating on the test set.
    #reloading to guaranteee the evaluation uses the exact saved weights and the inference-ready state (e.g., correct batch-norm statistics) that will be used later during analysis. 
    best_val_acc = max(history.history["val_accuracy"])
    epochs_trained = len(history.history["accuracy"])

    model.save(MODEL_PATH)
    del model
    keras.backend.clear_session()

    model = keras.models.load_model(MODEL_PATH)
    test_loss, test_acc = model.evaluate(test_ds_oh, verbose=0)

    print(f"\n{fold_key} results:")
    print(f"  Test accuracy:     {test_acc:.4f}")
    print(f"  Best val accuracy: {best_val_acc:.4f}")
    print(f"  Epochs trained:    {epochs_trained}")

    config_results[fold_key] = {
        "test_accuracy": float(test_acc),
        "test_loss": float(test_loss),
        "best_val_accuracy": float(best_val_acc),
        "epochs_trained": epochs_trained,
        "lr": LR,
        "unfrozen_layers": UNFROZEN_LAYERS,
        "dropout_1": DROPOUT_1,
        "dropout_2": DROPOUT_2,
    }
    with open(RESULTS_LOG, "w") as f:
        json.dump(config_results, f, indent=2)
    print(f"  Logged to {RESULTS_LOG}")

    del base_model
    keras.backend.clear_session()

print(f"\nConfiguration {CONFIG_IDX} finished")

if len(config_results) == 5:
    test_accs = [config_results[f"fold_{i}"]["test_accuracy"] for i in range(5)]
    print(f"Test accuracies per fold: {[f'{a:.4f}' for a in test_accs]}")
    print(f"Mean test accuracy: {np.mean(test_accs):.4f}")
    print(f"Std test accuracy:  {np.std(test_accs):.4f}")
else:
    print(f"Completed folds: {sorted(config_results.keys())}")
    print("Still missing folds to complete the configuration.")

Configuration 1: config_1_lr3e-05_lay40_drop0p4_0p5
  LR:              3e-05
  Unfrozen layers: 40
  Dropout:         (0.4, 0.5)

Already completed folds: ['fold_0', 'fold_1', 'fold_2', 'fold_3', 'fold_4']

Loading synAug from: /content/drive/MyDrive/augmentation_data/tank_classifier_v3.keras
synAug model loaded. Layers: 6
Found synAug base: efficientnetb4
Dense(128): dense
Output Dense(3): dense_1
synAug weights saved; original model dereferenced.

Skipping fold_0 (already completed)

Skipping fold_1 (already completed)

Skipping fold_2 (already completed)

Skipping fold_3 (already completed)

Skipping fold_4 (already completed)

Configuration 1 finished
Test accuracies per fold: ['0.7399', '0.6959', '0.7836', '0.7368', '0.7176']
Mean test accuracy: 0.7348
Std test accuracy:  0.0291
